# Lesson 6: One controlled session with the physical key already ON

**This notebook can produce a laser beam.** Use an attended, operator-approved
setup with cooling, interlocks, beam termination and suitable eyewear confirmed.
The physical key remains ON throughout; no key cycling or rapid key operation
is required. The diode current may already be ramped up and the laser enabled.
A closed safety shutter does not mean the diode current is zero.

Start with the safety shutter closed, clear active faults and locked temperature
servos. The reported laser state may be ON (1) or remote standby (0). Capture
fault history, store and verify the approved setpoint, then confirm RUN. The
emission cell skips L=1 when already ON, opens the safety shutter once, reads
power, then closes it and leaves the laser ON without moving the physical key.
Fault state 2 stops the workflow; never reset a fault merely to continue.

Run cells individually with one serial owner and prompt enabled. Writes use
the named controller setters; state checks retain byte payloads from `read()`.
All exchanges consume the complete reply CR/LF and enabled trailing prompt.
L=1 clears fault history and is used only when a fault-free laser is in remote standby.

**Stop after any error.** The laser may already be enabled. Use the physical
abort procedure when needed. There are no automatic retries or cleanup writes
over an uncertain link. Disconnecting is not an emergency stop.

## 1. Connection and approved power settings

Confirm port, model and baud with the operator. The checks below compare the
no-active-fault reply to `b"SYSTEM OK"`; other replies stop the sequence.
Use an operator-approved TARGET_W in 0.01 W increments and a site-approved
POWER_LIMIT_W in watts. The physical key must already be ON with shutter closed.
Restart the kernel after package updates.

With `prompt_enabled=True`, the controller reads the reply through CR/LF, then
consumes the enabled trailing prompt before sending another instruction. The
instrument prompt must already be enabled; construction sends no echo or prompt
changes. Matching instruction echoes and recognized leading notices are removed
from `read()` payloads while their received bytes remain in transaction evidence.

`TIMEOUT_S` configures serial write and receive waits. A reply line, each leading
notice and the trailing prompt use separate `read_until()` calls with that
configured timeout; there is no combined transaction deadline. No extra pause
is inserted between queries. Restart the kernel after a package update.

Saved outputs and execution counts are retained from earlier hardware runs;
they have not been regenerated for these revised cells.

In [1]:
PORT = "COM5"
MODEL = "V2"  # V2, V5 or V6
BAUDRATE = 19200
TIMEOUT_S = 2.0  # Serial write and receive timeout
TARGET_W = 0.02  # Operator-approved watts, in 0.01 W increments
POWER_LIMIT_W = 0.05  # Site-approved ceiling in watts

## 2. Validate settings before opening the port

Require a finite target in 0.01 W increments before opening the port.
Both the original target and the four-decimal transmitted value must be within
the ceiling. The bytes below preview the command and expected readback;
`set_power_w()` performs the write later and validates its acknowledgment.
Construction sends nothing. Writes require explicit enablement.

In [2]:
from math import isfinite, ulp

from coherent_verdi import VerdiController

if PORT == "<PORT>" or POWER_LIMIT_W is None:
    raise ValueError("Fill in every setting in section 1 first.")
laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    prompt_enabled=True,
    allow_writes=True,
    power_limit_w=POWER_LIMIT_W,
)
if type(TARGET_W) not in (int, float) or not isfinite(TARGET_W):
    raise ValueError("TARGET_W must be a finite number.")
if not 0 <= TARGET_W <= laser.power_limit_w:
    raise ValueError("TARGET_W must be between 0 and POWER_LIMIT_W.")
stepped_target = round(TARGET_W, 2)
if abs(TARGET_W - stepped_target) > 2 * ulp(TARGET_W):
    raise ValueError("TARGET_W must be in 0.01 W increments (e.g. 0.01, 0.02, 0.03).")
target_bytes = b"%.4f" % stepped_target
sent_w = float(target_bytes)
if not 0 <= sent_w <= laser.power_limit_w:
    raise ValueError("Rounded TARGET_W exceeds POWER_LIMIT_W.")
power_command = b"P=" + target_bytes
print("Will send:", repr(power_command))

Will send: b'P=0.0200'


## 3. Reply validation and evidence

`read()` returns payload `bytes` with echo, CR/LF framing, recognized leading
firmware notices and prompt removed. It does not convert the payload to numbers,
booleans or fault codes. Bytes returned by serial reads, including notices and incomplete replies returned
at a timeout, remain in `laser.last_transaction["response"]`.

Named setters validate their acknowledgments through the controller parser.
Recognized notices include `min current set`, `level0 set!` and `level1 set!`;
their wire bytes remain in `laser.last_transaction["response"]`.
The notebook does not duplicate the parser or retry failed exchanges.

## 4. Open the serial port

Run this cell by itself and inspect the result before starting serial communication.
`connect()` opens the local serial port and returns `True` on success. Failures
raise an exception. It sends no instrument instructions. A successful open does
not establish that the laser is responding or ready.

Close any other serial owner first. If opening fails, stop here.

`override=True` closes an older controller connection to the same port in this
Python process before opening the new one. It cannot take ownership from another
kernel/application or a raw `ser` handle; release those explicitly. A failed
controller still requires a fresh controller object. Reopening does not establish
instrument synchronization or repair corrupted replies.

In [3]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

Serial port open: True


## 5. Start serial communication

Only after inspecting the connection result, run the following cells separately.
Buffer maintenance and instrument queries belong here, outside the connection
section. An explicit purge discards host queues; it does not reset the device's
parser. No delay or purge is claimed to prove synchronization. Each query consumes
the complete reply CR/LF and enabled trailing prompt before another can be sent.

In [4]:
laser.purge_buffers()
replies = []
acknowledgments = []


def read_checked(query):
    payload = laser.read(query)
    replies.append((query, payload))
    print(query, repr(payload))
    if not payload or payload.startswith((b"RANGE ERROR:", b"Command Error:", b"Query Error:")):
        raise RuntimeError(f"Unexpected query reply: {payload!r}")
    return payload


def record_acknowledgment():
    transaction = laser.last_transaction
    instruction, wire = transaction["instruction"], transaction["response"]
    acknowledgments.append((instruction, wire))
    print(instruction, "acknowledgment:", repr(wire))


version_reply = read_checked(b"?SV")

b'?SV' b'6.05'


## 6. Check readiness with the physical key already ON

Require ?K = 1, reported laser ON (1) or remote standby (0), shutter closed,
verified clear active faults and all four temperature servos LOCKED. This cell
only reads; it does not interrupt an already enabled laser or its diode current.
State 2 is faulted and stops preparation even if the active-fault text looks clear.
Checks are sequential and cannot prevent external state changes between requests.

In [7]:
def check_ready():
    state = {query: read_checked(query) for query in (b"?K", b"?L", b"?S", b"?F")}
    if state[b"?K"] != b"1":
        raise RuntimeError("Require physical key already ON throughout this notebook.")
    if state[b"?S"] != b"0" or state[b"?F"] != b"SYSTEM OK":
        raise RuntimeError("Require shutter closed and no active faults.")
    if state[b"?L"] not in (b"0", b"1"):
        raise RuntimeError("Require laser ON or remote standby; do not reset a faulted state.")
    for query in (b"?D1SS", b"?LBOSS", b"?ESS", b"?VSS"):
        if read_checked(query) != b"1":
            raise RuntimeError(f"Temperature servo {query!r} is not LOCKED; do not proceed.")
    return state


check_ready()
print(
    "Reported readiness: key ON, laser ON or remote standby, shutter closed, "
    "no active faults, servos locked."
)

b'?K' b'1'
b'?L' b'1'
b'?S' b'0'
b'?F' b'SYSTEM OK'
b'?D1SS' b'1'
b'?LBOSS' b'1'
b'?ESS' b'1'
b'?VSS' b'1'
Reported readiness: key ON, laser ON or remote standby, shutter closed, no active faults, servos locked.


## 7. Preserve current fault history

The laser may already be enabled. Capture the fault-history payload before
any command from this notebook; the full wire reply remains in
`laser.last_transaction`. If it is in remote standby, the later L=1 clears history; when already ON, no L=1 is sent. A clear history alone does not establish
readiness. Never enable as a fault-reset or recovery action.

In [8]:
history_before_enable = laser.read(b"?FH")
print("Fault history before enable:", repr(history_before_enable))
if not history_before_enable or history_before_enable.startswith(
    (b"RANGE ERROR:", b"Command Error:", b"Query Error:")
):
    raise RuntimeError(f"Unexpected fault history reply: {history_before_enable!r}")

Fault history before enable: b'SYSTEM OK'


## 8. Set and verify power with the shutter closed

The physical key remains ON; the laser may already be enabled. Type SET to store
the approved setpoint. Recheck readiness immediately before the write.
`set_power_w()` changes the setpoint and validates its acknowledgment; it does not
send a standby or enable command. Retain the full reply in acknowledgments.
Require a matching four-decimal ?SP byte payload, then check readiness again.
A mismatch or failed check stops before this notebook opens the shutter.

In [9]:
if input("To store the setpoint, type SET: ").strip() != "SET":
    raise RuntimeError("Cancelled. No settings were changed.")
check_ready()
laser.set_power_w(TARGET_W)
record_acknowledgment()
if read_checked(b"?SP") != target_bytes:
    raise RuntimeError("Setpoint readback mismatch; do not enable.")
check_ready()

b'?K' b'1'
b'?L' b'1'
b'?S' b'0'
b'?F' b'SYSTEM OK'
b'?D1SS' b'1'
b'?LBOSS' b'1'
b'?ESS' b'1'
b'?VSS' b'1'
b'P=0.0200' acknowledgment: b'\r\nVerdi V-2>'
b'?SP' b'0.0200'
b'?K' b'1'
b'?L' b'1'
b'?S' b'0'
b'?F' b'SYSTEM OK'
b'?D1SS' b'1'
b'?LBOSS' b'1'
b'?ESS' b'1'
b'?VSS' b'1'


{b'?K': b'1', b'?L': b'1', b'?S': b'0', b'?F': b'SYSTEM OK'}

## 9. Run one shutter cycle

Type RUN to check readiness, open the shutter, read power and close the shutter.
The laser and physical key stay ON; no standby command is sent. If already ON,
the laser is not enabled again. Stop after any error and inspect section 10.

In [10]:
def verify_readbacks(expected):
    actual = {query: read_checked(query) for query in expected}
    if actual != expected:
        raise RuntimeError(f"Readback mismatch: expected {expected!r}, received {actual!r}")
    return actual


if input("Type RUN for one shutter cycle; the laser will remain ON: ").strip() != "RUN":
    raise RuntimeError("Cancelled. No emission commands sent; the laser may already be enabled.")
state = check_ready()
verify_readbacks({b"?SP": target_bytes})
if state[b"?L"] == b"0":
    laser.set_laser_state(1)
    record_acknowledgment()
verify_readbacks({b"?L": b"1", b"?F": b"SYSTEM OK", b"?S": b"0"})

laser.set_shutter(open=True)
record_acknowledgment()
verify_readbacks({b"?S": b"1", b"?L": b"1", b"?F": b"SYSTEM OK"})
power_reply = laser.read(b"?P")
print("Reported power while open (raw):", repr(power_reply))
if not power_reply or any(marker in power_reply for marker in (b"Error:", b"ERROR:")):
    raise RuntimeError(f"Unexpected power reply: {power_reply!r}")

laser.set_shutter(open=False)
record_acknowledgment()
final_payloads = verify_readbacks(
    {b"?K": b"1", b"?L": b"1", b"?S": b"0", b"?F": b"SYSTEM OK", b"?SP": target_bytes}
)
print("Shutter CLOSED; laser ON; physical key ON; setpoint unchanged.")

b'?K' b'1'
b'?L' b'1'
b'?S' b'0'
b'?F' b'SYSTEM OK'
b'?D1SS' b'1'
b'?LBOSS' b'1'
b'?ESS' b'1'
b'?VSS' b'1'
b'?SP' b'0.0200'
b'?L' b'1'
b'?F' b'SYSTEM OK'
b'?S' b'0'
b'S=1' acknowledgment: b'\r\nVerdi V-2>'
b'?S' b'1'
b'?L' b'1'
b'?F' b'SYSTEM OK'
Reported power while open (raw): b'0.025'
b'S=0' acknowledgment: b'\r\nVerdi V-2>'
b'?K' b'1'
b'?L' b'1'
b'?S' b'0'
b'?F' b'SYSTEM OK'
b'?SP' b'0.0200'
Shutter CLOSED; laser ON; physical key ON; setpoint unchanged.


## 10. Inspect an error

Inspect the last transaction and retained `replies`/`acknowledgments` without
sending anything. Use the physical abort procedure if needed before releasing
the port. Do not retry an uncertain write or send automatic cleanup commands.

In [11]:
print(laser.last_transaction)

{'instruction': b'?SP', 'request': b'?SP\r\n', 'response': b'0.0200\r\nVerdi V-2>', 'payload': b'0.0200', 'outcome': 'received', 'error': None}


## 11. Disconnect

Release the serial port. Normal completion above has verified the shutter
CLOSED and the laser still ON. Disconnect sends no laser or shutter commands.
Full shutdown is a separate operator procedure.

In [12]:
laser.disconnect()